In [ ]:
#SETUP - Install Groq Library
# API KEY : https://console.groq.com/keys

#step 1 :install the dependencies

!pip install groq --quiet

import os   # it needs to communicate with local files
import json
import re
import time
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("All Libraries imported successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.5 MB/s eta 0:00:00
All Libraries imported successfully


In [ ]:
# configure your groiq API

from groq import Groq
API_KEY="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"
client = Groq(api_key=API_KEY)
MODEL="llama-3.1-8b-instant"
print("Groq Client configured with model:",MODEL)


Groq Client configured with model: llama-3.1-8b-instant


In [ ]:
## your first LLM API call

def ask_llm(user_message,system_message="You are helpful assistant",
            temperature=0.7,max_tokens=500):
      """

      Send a message to the LLM and return the required text

      Parameters:
      -----------
      error_message: give an proper prompt
      user_message : Instruction about role and behaviour
      temperature : 0.0-deterministic,1.0=creative
      max_tokens: Maximum response length

      """
      response=client.chat.completions.create(
          model=MODEL,
          messages=[{
              "role":"system",
              "content":system_message
          },
          {
              "role":"user",
              "content":user_message
          }],
          temperature=temperature,
          max_tokens=max_tokens,

      )
      return response.choices[0].message.content

test_response=ask_llm("What is the meaning of indhuja ?")
print('=== RESPONSE ===\n',test_response)

=== RESPONSE ===
 Indhuja is a Sanskrit name, derived from the word 'Indhu' which means 'sacred, divine'. It is also associated with the word 'Indra', who is the god of the sky and thunder in Hindu mythology.

In Sanskrit, 'Indhuja' can be interpreted as 'daughter of Indra' or 'divine child'. It is a name often used in Hindu mythology and literature to refer to a female deity or a goddess associated with Indra.

In modern times, Indhuja is also used as a given name, particularly in India and other countries with significant Hindu populations. It is often associated with qualities such as strength, courage, and divine grace.

In some contexts, Indhuja can also refer to a specific deity or goddess from Hindu mythology, often depicted as a female companion of Indra or a goddess of fertility and prosperity.

Overall, the meaning of Indhuja is closely tied to its roots in Hindu mythology and Sanskrit language, and it is often associated with qualities of divine strength and beauty.


In [ ]:
### Understand tokens and context

# Ask about concepts from Phase 1 - showing LLM knows details our domain

response_url = ask_llm(
    " In a 3 bullet points,explain how the Medallian Architecture."
    "How (broze,Silver,Gold layers) relates to ETL pipelines?",
    system_message="You are a senior Data engineer instructor."
    "Be consice and practical"
)
print("Medallian <- ETL connection")
print(response_url)
print()
print("=== Token Explanation ===")
print("Each word is roughly 1-2 tokens")
print('The model above used approximately',len(response_etl.split())*1.3,'tokens.')
print('Llama-3.1-8b context window: 8192 tokens(~6000 words per conversation)')

Medallion+ETL connection:
Here are 3 bullet points explaining how the Medallion Architecture relates to ETL pipelines:

• **Data Ingestion (Bronze Layer)**: This layer is responsible for ingesting raw data from various sources into a centralized data lake or store. The ETL pipeline in this layer focuses on data extraction, loading, and storing raw data. It's the initial step in processing and storing data.

• **Data Processing (Silver Layer)**: Once the raw data is ingested, the Silver layer applies additional processing and quality checks to the data. The ETL pipeline in this layer focuses on data transformation, data validation, and data enrichment. It's where data is prepared for analysis and reporting.

• **Data Analytics (Gold Layer)**: The Gold layer is where data is analyzed and reported to provide insights and business value. The ETL pipeline in this layer focuses on data aggregation, reporting, and visualization. It's the final step in processing data to support business decis

In [ ]:
# Zero Line Prompt

zero_shot_response = ask_llm(
    "Extract the city name from this address."
    "77H Church Street,Pallapalayam,Tiruppur,642663,Tamil Nadu"
)
print("===  ZERO SHOT RESPONSE ===")
print(zero_shot_response,'\n')

ambiguous_response = ask_llm("Clen this data: Indhuja,200000,Tirunelveli")
print("===  AMBIGUOUS RESPONSE ===")
print(ambiguous_response,'\n')

===  ZERO SHOT RESPONSE ===
The city name extracted from the given address is: 

- Tiruppur 

(Pallapalayam is a locality or area within Tiruppur) 

===  AMBIGUOUS RESPONSE ===
The data provided appears to be a record in a database or a spreadsheet. It contains three fields:

1. Name: Indhuja
2. Salary: 200000
3. Location: Tirunelveli

If you'd like, I can help you reformat or organize this data in any way. Please let me know what you're trying to achieve. 



In [ ]:
df=pd.read_csv("/content/drive/MyDrive/DAY 1/student_performance.csv")

Gen AI : it will predict the output even i didn't give the input based on the past conversations

In [ ]:
zero_shot_response_csv = ask_llm(
    f"Summarize the following student performance data in bullet points:\n{df.head().to_csv(index=False)}",
    system_message="You are a data analyst. Provide a concise summary of the data."
)
print("=== ZERO SHOT RESPONSE FROM CSV DATA ===")
print(zero_shot_response_csv)

=== ZERO SHOT RESPONSE FROM CSV DATA ===
Here's a concise summary of the student performance data in bullet points:

**Overall Statistics:**

* Average age: 19.6 years
* Average semester score (math, science, english, programming): 78.8
* Average attendance percentage: 90.5

**Department-wise Statistics:**

* Computer Science: 3 students, average math score: 84.3, average programming score: 88.3
* Electronics: 1 student, average math score: 65, average programming score: 55
* Mechanical: 1 student, average math score: 70, average programming score: 48

**Student-wise Statistics:**

* Aarav Sharma (1001): high programming score (91), high attendance percentage (92)
* Priya Patel (1002): high science score (82), high english score (88)
* Rohit Verma (1003): low programming score (55), low math score (65)
* Sneha Reddy (1004): high attendance percentage (95), low programming score (48)
* Arjun Nair (1005): high math score (92), high programming score (95)

**City-wise Statistics:**

* Mum

In [ ]:
zero_shot_extract_column = ask_llm(
    f"Extract only the 'city' column from the following student performance data. Provide the names as a comma-separated list:\n{df.head().to_csv(index=False)}",
    system_message="You are a data extraction bot. Only provide the requested data, nothing else."
)
print("=== EXTRACTED NAME COLUMN ===")
print(zero_shot_extract_column)

=== EXTRACTED NAME COLUMN ===
Mumbai, Ahmedabad, Delhi, Hyderabad, Kochi


In [ ]:
## few shot prompt

few_shot_prompt="""
Converts employee text to JSON,Here are example:

Input :Indhuja,20000,Tirunelveli
Output:{"name":"Indhuja","salary":"20000","city":"Tirunelveli"}

Input:Jo,100000,London
Output:{"name":"Jo","salary":"100000","city":"London"}

Input:Fazzzzzzzzzzzzz,300000,USA
Output:{"name":"Fazzzzzzzzzzzzz","salary":"300000","city":"USA"}
"""

few_shot_response = ask_llm(few_shot_prompt, temperature=0.8)
print("==== Few Shot Result====")
print(few_shot_response,'\n')

try:
  parsed = json.loads(few_shot_response.strip())
  print("Successfully parsed as JSON")
  print(f"Name: {parsed['name']}, Salary: {parsed['salary']}, City: {parsed['city']}")
except json.JSONDecodeError:
  print("Parsing failed - model added extra text")
  print("Solution: add explicit instructions in the system prompt")

==== Few Shot Result====
Here's a simple Python function that can achieve this:

```python
def convert_to_json(employee_str):
    """
    Converts a string of employee information to a JSON object.

    Args:
        employee_str (str): A string containing employee information in the format 'name,salary,city'

    Returns:
        dict: A JSON object containing the parsed employee information
    """
    employee_info = employee_str.split(',')
    return {
        "name": employee_info[0],
        "salary": employee_info[1],
        "city": employee_info[2]
    }

# Example usage:
print(convert_to_json('Indhuja,20000,Tirunelveli'))
# Output: {'name': 'Indhuja', 'salary': '20000', 'city': 'Tirunelveli'}

print(convert_to_json('Jo,100000,London'))
# Output: {'name': 'Jo', 'salary': '100000', 'city': 'London'}

print(convert_to_json('Fazzzzzzzzzzzzz,300000,USA'))
# Output: {'name': 'Fazzzzzzzzzzzzz', 'salary': '300000', 'city': 'USA'}
```

This function splits the input string into a list

In [ ]:
## few shot prompt

few_shot_prompt="""
Tells the gender of a person by name,

Input :Indhuja
Output:Female

Input:Harish
Output:Male

Input:Jothika
Output:Female

who is the fazmina?
"""

few_shot_response = ask_llm(few_shot_prompt, temperature=0.8)
print("==== Few Shot Result====")
print(few_shot_response,'\n')

try:
  parsed = json.loads(few_shot_response.strip())
  print("Successfully parsed as JSON")
  print(f"Name: {parsed['name']}, Salary: {parsed['salary']}, City: {parsed['city']}")
except json.JSONDecodeError:
  print("Parsing failed - model added extra text")
  print("Solution: add explicit instructions in the system prompt")

==== Few Shot Result====
Based on the input names, I can tell the gender of the person. However, please note that this is not 100% accurate, as some names can be ambiguous or have multiple associations.

Here are the outputs:

- Indhuja: Female (In Hindu mythology, Indhuja is the daughter of Lord Shiva)
- Harish: Male (Harish is a common masculine name in many Indian cultures)
- Jothika: Female (Jothika is a feminine name in Tamil, which is commonly used in South India)

Now, about Fazmina... I couldn't find any information on a person named Fazmina. This could be a rare or unknown name, or it could be a misspelling or variation of a different name. Could you provide more context or information about Fazmina? 

Parsing failed - model added extra text
Solution: add explicit instructions in the system prompt


In [ ]:
# WITHOUT RULE

same_question = "Review this Python code and identify any issues\n"\
"df[total] = df[price] * df[quantity]\n"\
"print(df.groupby('product').sum())\n"

generic_response=ask_llm(same_question,temperature=0.2)
print('without Role Prompting')
print(generic_response[:300],'...')
print()

#WITH ROLE
role_response=ask_llm(same_question,
                      system_message="You are senior data engineer with 10 years of production"
                      "experience. REview code critically for production readiness,"
                      "data types issues,and potential failures at scale.",
                      temperature=0.2)
print('with Role Prompting (Senior Data Engineer):')
print(role_response[:400],'...')
print()
print('Notice:role prompting produces more technical,actionable feedback')

without Role Prompting
The provided Python code appears to be a part of a data analysis task using the pandas library. However, there are a few potential issues that can be identified:

1. **Undefined variables**: The code uses variables `df`, `total`, `price`, and `quantity`, but it's not clear where these variables are  ...

with Role Prompting (Senior Data Engineer):
**Code Review**

The provided Python code appears to be a simple data manipulation task using the pandas library. However, there are several potential issues that could impact production readiness:

### 1. Data Type Issues

The code assumes that the `price` and `quantity` columns are numeric, but it does not check for this. If either of these columns contains non-numeric data, the multiplication o ...

Notice:role prompting produces more technical,actionable feedback


In [ ]:
prompt="Give me one creation name for a data analytics startup"
print('==== Temperature Experiment ====')
for temp in[0.0,0.5,1.0]:
  response=ask_llm(prompt,temperature=temp)
  print(f'Temperature={temp}: {response.strip()}')
  time.sleep(1)  #small pause to respect rate limits

print()
print("Observations")
print(" temperature=0.0 -> same or very similar answer run(deterministic)")
print(" temperature=0.5 -> some variations")
print(" temperature=1.0 -> more creative and varied,sometimes surprising ")
print()
print("Rule for data engineering tasks:use temperature=0.0 or 0.1")
print('You need CONSISTENT,PARSEABLE output = not creative variations')

==== Temperature Experiment ====
Temperature=0.0: Here's a potential creation name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connection and linkages, implying the ability to connect data points and provide insights. "Insights" clearly communicates the focus on data analysis and interpretation.
Temperature=0.5: Here's a suggestion for a data analytics startup name:

**Nexia Insights**

"Nexia" implies connection and integration, which is fitting for a data analytics startup that helps organizations connect their data sources and gain insights from them. The name also has a modern and tech-savvy feel to it, which could appeal to potential customers.
Temperature=1.0: Here's a potential name for a data analytics startup:

"Insighterra"

This name suggests a connection to insights, data, and possibly even earth or ground-level understanding, which can be appealing for a data analytics firm.

Observations
 temperature=0.0 -> same or very similar answer run(determinist

In [ ]:
messy_invoices = [
    "INV-2024-0091 TECHWORLD SOLUTIONS 15th Jan 2024 Rs.45,000 Laptop purchase",
    "Invoice from PRIYA ENTIREPRICES dt 07-02-2024 amt: 12500 for Office Cleaning services",
    "#INV-2024-103 | arjun nair consultancy | 5000 | march 15 2024 | python training",
    "SURESH RAO HARDWARE STORE 2500 Keyboard and mouse accessories 2024/01/10",
    "Tax Invoice:Ananya Tech Solutions | Inv-897|Date:28-feb-24 | Amount:INR 95,000|Server hardware"
]

print('Messy invoices to process:')
for i ,inv in enumerate(messy_invoices, 1):
  print(f'{i+1}.{inv}')
print(f'\nTotal: {len(messy_invoices)}invoices')

Messy invoices to process:
2.INV-2024-0091 TECHWORLD SOLUTIONS 15th Jan 2024 Rs.45,000 Laptop purchase
3.Invoice from PRIYA ENTIREPRICES dt 07-02-2024 amt: 12500 for Office Cleaning services
4.#INV-2024-103 | arjun nair consultancy | 5000 | march 15 2024 | python training
5.SURESH RAO HARDWARE STORE 2500 Keyboard and mouse accessories 2024/01/10
6.Tax Invoice:Ananya Tech Solutions | Inv-897|Date:28-feb-24 | Amount:INR 95,000|Server hardware

Total: 5invoices


In [ ]:
extracted_records = []

for invoice_text in messy_invoices:
    prompt = f"""Extract the invoice ID, company name, invoice date (YYYY-MM-DD), amount, and description from the following invoice text. Return the information as a JSON object with keys: invoice_id, company_name, invoice_date, amount, description. Convert amount to a float and remove currency symbols. For date, extract the date (e.g. 15th Jan 2024 becomes 2024-01-15, 07-02-2024 becomes 2024-02-07, March 15 2024 becomes 2024-03-15, 2024/01/10 becomes 2024-01-10, 28-feb-24 becomes 2024-02-28). If an invoice ID is not explicitly mentioned, generate one based on the company name and date (e.g., TECHWORLD-2024-01-15). If no explicit invoice ID, try to find a pattern like INV-YYYY-XXX.

Invoice text: {invoice_text}"""

    # Use a low temperature for deterministic output
    llm_response = ask_llm(
        prompt,
        system_message="You are an expert at extracting structured information from unstructured text. ONLY return the JSON object, with keys: invoice_id, company_name, invoice_date, amount, description. Ensure all amounts are floats and dates are in YYYY-MM-DD format. Do not include any conversational text or code blocks.",
        temperature=0.0
    )
    try:
        # Use regex to extract potential JSON string from markdown code blocks or assume direct JSON
        json_match = re.search(r'```(?:json|python)?\s*({.*})\s*```', llm_response, re.DOTALL)
        if json_match:
            cleaned_response = json_match.group(1)
        else:
            cleaned_response = llm_response.strip() # Assume it's direct JSON if no code block markers
        record = json.loads(cleaned_response)
        extracted_records.append(record)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON for invoice: {invoice_text}\nResponse: {llm_response}\nError: {e}")

print("=== EXTRACTED INVOICE RECORDS ===")
for record in extracted_records:
    print(record)

=== EXTRACTED INVOICE RECORDS ===
{'invoice_id': 'INV-2024-0091', 'company_name': 'TECHWORLD SOLUTIONS', 'invoice_date': '2024-01-15', 'amount': 45000.0, 'description': 'Laptop purchase'}
{'invoice_id': 'PRIYA-2024-02-07', 'company_name': 'PRIYA ENTIREPRICES', 'invoice_date': '2024-02-07', 'amount': 12500.0, 'description': 'Office Cleaning services'}
{'invoice_id': 'INV-2024-103', 'company_name': 'arjun nair consultancy', 'invoice_date': '2024-03-15', 'amount': 5000.0, 'description': 'python training'}
{'invoice_id': 'SURESH-2024-01-10', 'company_name': 'SURESH RAO HARDWARE STORE', 'invoice_date': '2024-01-10', 'amount': 2500.0, 'description': 'Keyboard and mouse accessories'}
{'invoice_id': 'INV-2024-28', 'company_name': 'Ananya Tech Solutions', 'invoice_date': '2024-02-28', 'amount': 95000.0, 'description': 'Server hardware'}


### Manually Push Notebook to GitHub

Follow these steps to manually push your notebook to your GitHub repository if Colab's direct saving fails. Ensure you have manually saved or downloaded your `.ipynb` file to your Colab environment's working directory before proceeding. We will assume the notebook file is named `Day6_ipynb.ipynb`.

In [ ]:
# 1. Configure Git with your user name and email
# Replace with your GitHub username and email
!git config --global user.email "your-email@example.com"
!git config --global user.name "Your GitHub Username"

print("Git configured successfully!")

Git configured successfully!


### Important Security Note for Cloning Private Repositories

If your repository is **private**, you will need to use a [GitHub Personal Access Token (PAT)](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/creating-a-personal-access-token) instead of your password. Create a PAT with `repo` scope, and then use it like this when prompted or directly in the URL:

`!git clone https://<YOUR_PAT>@github.com/INDHUJA007-HUB/indhuja-day15-workshop.git`

For **public** repositories, direct cloning should work without a PAT.

In [ ]:
# 2. Clone your repository
# This will create a directory named 'indhuja-day15-workshop'
!git clone https://github.com/INDHUJA007-HUB/indhuja-day15-workshop.git

# Navigate into the cloned repository
%cd indhuja-day15-workshop
print("Repository cloned and directory changed!")

Cloning into 'indhuja-day15-workshop'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 111 (delta 35), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 236.00 KiB | 7.61 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/indhuja-day15-workshop
Repository cloned and directory changed!


In [ ]:
# 3. Copy your notebook into the repository
# Assuming your notebook is named 'Day6_ipynb.ipynb' and is in the /content/ directory.
# Adjust the source path if your notebook file is in a different location.
# The target path 'Day6/' is derived from your error message. Create it if it doesn't exist.

notebook_filename = 'Day6_ipynb.ipynb' # Ensure this matches your downloaded/saved notebook name
source_path = f'/content/{notebook_filename}' # Adjust if you saved it elsewhere
target_directory_in_repo = 'Day6'

!mkdir -p {target_directory_in_repo} # Ensure the target directory exists
!cp {source_path} {target_directory_in_repo}/{notebook_filename}

print(f"'{notebook_filename}' copied to '{target_directory_in_repo}' in the repository!")

cp: cannot stat '/content/Day6_ipynb.ipynb': No such file or directory
'Day6_ipynb.ipynb' copied to 'Day6' in the repository!


In [ ]:
# 4. Add, commit, and push your changes

!git add .
!git commit -m "Update Day6_ipynb.ipynb from Colab (manual push)"

# Push to the 'main' branch
# If your default branch is different (e.g., 'master'), change 'main' below.
# You might be prompted for your GitHub username and password/PAT.
!git push origin main

print("Changes pushed to GitHub!")

# Go back to the root content directory (optional)
%cd /content/


On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
fatal: could not read Username for 'https://github.com': No such device or address
Changes pushed to GitHub!
/content


In [ ]:
!ls -F /content/

In [ ]:
prompt = """ Extract the name, age, and salary from the following text and convert them into a JSON object.

     the TEXT is: Rahul,45,100000
"""


In [ ]:
import pandas as pd

# extracted_records will be populated by the previous cell

invoices_df=pd.DataFrame(extracted_records)

invoices_df['invoice_data']=pd.to_datetime(invoices_df['invoice_date'], errors='coerce')

print('=== SMART DATA CLEANER OUTPUT ===')
print(f'Rows: {len(invoices_df)} | Columns: {len(invoices_df.columns)}')
print()
print(invoices_df.to_string(index=False))

=== SMART DATA CLEANER OUTPUT ===
Rows: 5 | Columns: 5

invoice_date              company_name  amount                    description invoice_data
  2024-01-15       TECHWORLD SOLUTIONS 45000.0                Laptop purchase   2024-01-15
  2024-02-07        PRIYA ENTIREPRICES 12500.0       Office Cleaning services   2024-02-07
  2024-03-15    ARJUN NAIR CONSULTANCY  5000.0                Python training   2024-03-15
  2024-01-10 SURESH RAO HARDWARE STORE  2500.0 Keyboard and mouse accessories   2024-01-10
  2024-02-28     ANANYA TECH SOLUTIONS 95000.0                Server hardware   2024-02-28
